# London Transport Network 

Models the London contactless network as a graph to find the most likely interchange points between DLR and other services based on shortest journey time. 

### Key Concepts:
* **NLC (National Location Code):** Represents a set of physical ticket gates. Separates multi-modal stations (e.g., London Bridge NR vs. Underground) into distinct graph nodes.

* **ICS:** TfL’s station-level identifier used for Journey Planner API lookups.

* **NaPTAN (National Public Transport Access Node)**: A national standard system identifier used across the UK transport network to provide precise station-level granularity. The code is typically prefixed with 940GZZLU for Tube, 940GZZDL for DLR, or 910G/490G for National Rail stations.

**Output:** A comprehensive mapping table of all DLR-to-non-DLR journeys with optimal in-station interchange points.

In [1]:
# ==============================================================================
# LIBRARY IMPORTS & ENVIRONMENT SETUP
# ==============================================================================

# File management, environment variables, and graph serialization
import os
import pickle
from dotenv import load_dotenv

# Tabular data manipulation and numerical operations
import numpy as np
import pandas as pd

# Python built-ins for data processing and rate limiting
from itertools import combinations
import json
import re
import time

# Querying TfL API endpoints
import requests
import urllib

# Constructing network topology and running Dijkstra shortest path algotrithm
import networkx as nx

# Load API key from environment (.env file)
load_dotenv()
TFL_API_KEY = os.getenv("TFL_API_KEY", "")

if not TFL_API_KEY:
  print(
      "Warning: TFL_API_KEY not found in environment variables. API calls will"
      " rely on unauthenticated rate limits."
  )
else:
  print("TFL_API_KEY successfully loaded.")

TFL_API_KEY successfully loaded.


## Network Nodes - All Contactless Stations

Loads all contactless train stations across National Rail, Tube, DLR, Overground, and Elizabeth line. <br>
_Bus, tram, and river services are excluded._

Each station row maps its unique **NLC** (gate line) to its corresponding **ICS** station identifier.

In [2]:
# Load station reference file
all_stations = pd.read_csv("all_stations_ref.csv")

# Filter out redundant lines (TfL Rail & Overground that are already included under different names)
excluded_lines = ["tfl-rail", "london-overground"]
all_stations = all_stations[~all_stations["Line"].isin(excluded_lines)]

all_stations["Node_ID"] = (
    all_stations["Station"].str.lower() + "_" + all_stations["Line"].str.lower()+ "_" + all_stations["Gate_NLC"].astype(str)
)

print(
    f"Loaded {all_stations['Node_ID'].nunique()} unique station/line/NLC pairings, "
    f"with {all_stations['Station'].nunique()} individual stations and"
    f" {all_stations['Gate_NLC'].nunique()} unique NLC gate lines."
)
all_stations.head()

Loaded 1242 unique station/line/NLC pairings, with 807 individual stations and 841 unique NLC gate lines.


,Station,NaPTAN,Line,NLC,Gate_NLC,Check,Node_ID
0,Abbey Road,940GZZDLABR,dlr,750,750,True,abbey road_dlr_750
1,Abbey Wood,9100ABWD0,elizabeth,5131,5131,True,abbey wood_elizabeth_5131
3,Abbey Wood,9100ABWD0,thameslink,5131,5131,True,abbey wood_thameslink_5131
4,Abbey Wood,910GABWD,southeastern,5131,5131,True,abbey wood_southeastern_5131
5,Abbey Wood,910GABWD,thameslink,5131,5131,True,abbey wood_thameslink_5131


## Network Edges: Adjacent Station Travel Times

### Data Source & Methodology:
* **Source:** Derived from TfL Line Route Sequence feeds and queried via the TfL Journey Planner API for active rail modes (Tube, DLR, Overground, Elizabeth Line and National Rail).

* **Assumptions & Fallbacks:** 
  * Only adjacent station pairs along active branch sequences are queried.
  * If a live API query fails or returns zero duration, a fallback travel time of `0.5` minutes (`FALLBACK_TRAVEL_TIME`) is applied and logged to `travel_times_investigation_log.csv`.

In [3]:
# ==============================================================================
# LINE TRAVEL TIMES BUILDER (HYBRID API + GJT MATRIX FALLBACK)
# ==============================================================================
# Queries TfL Line Route sequences and Journey Planner API, supplementing gaps 
# with the National Rail GJT matrix (`GJT.csv`) for complete network coverage.
# ==============================================================================

RAIL_MODES = ["tube", "dlr", "overground", "elizabeth-line", "national-rail"]
FALLBACK_TRAVEL_TIME = 0.5


def load_gjt_matrix():
  """Loads and indexes the GJT.csv matrix for quick NLC-to-NLC travel time lookups."""
  if not os.path.exists("GJT.csv"):
    print("Warning: 'GJT.csv' not found. Fallback matrix disabled.")
    return {}
  
  gjt_df = pd.read_csv("GJT.csv")
  gjt_lookup = {}
  for _, row in gjt_df.iterrows():
    start_nlc = int(row["Start_NLC"])
    end_nlc = int(row["End_NLC"])
    mins = float(row["GJT"]) / 100.0  # Convert GJT scale to minutes
    gjt_lookup[(start_nlc, end_nlc)] = mins
    
  print(f"Loaded {len(gjt_lookup):,} pairs into GJT fallback lookup matrix.")
  return gjt_lookup


def build_naptan_and_nlc_lookups():
  """
    Builds primary dictionaries from 'all_stations' mapping NaPTAN, Node_ID, and NLCs.
  """
  naptan_map = {}
  node_id_map = {}
  nlc_to_node_map = {}
  target_naptans = set()

  try:
    for _, row in all_stations.iterrows():
      station_name = str(row["Station"]).strip()
      naptan = str(row["NaPTAN"]).strip()
      node_id = str(row["Node_ID"]).strip()
      nlc = row["NLC"]

      if pd.notna(naptan) and naptan != "" and naptan != "nan":
        target_naptans.add(naptan)
        node_id_map[naptan] = node_id
        if pd.notna(nlc):
          nlc_int = int(nlc)
          nlc_to_node_map[nlc_int] = node_id

    print(f"Loaded {len(target_naptans):,} unique NaPTAN lookup mappings from all_stations table.")

  except Exception as e:
    print(f"Warning/Error processing 'all_stations' lookups: {e}")

  return naptan_map, node_id_map, nlc_to_node_map, target_naptans


def get_line_travel_times():
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
  gjt_lookup = load_gjt_matrix()
  _, node_id_map, _, target_naptans = build_naptan_and_nlc_lookups()

  # Step 1: Fetch active lines across specified rail modes
  lines_url = f"https://api.tfl.gov.uk/Line/Mode/{','.join(RAIL_MODES)}"
  res = requests.get(lines_url, params=params)
  if res.status_code != 200:
    print(f"Failed to fetch lines (Status {res.status_code}).")
    return pd.DataFrame()

  lines = res.json()
  records = []
  error_investigations = []

  print(f"Found {len(lines)} rail lines. Processing sequence pairs...")

  for line in lines:
    line_id = line["id"].lower()
    line_name = line["name"]

    print(f"  -> Processing line: {line_name}")

    # Step 2: Fetch route sequence branch stops
    seq_url = f"https://api.tfl.gov.uk/Line/{line_id}/Route/Sequence/all"
    seq_res = requests.get(seq_url, params=params)
    if seq_res.status_code != 200:
      continue

    seq_data = seq_res.json()

    for branch in seq_data.get("stopPointSequences", []):
      stops = branch.get("stopPoint", [])

      for i in range(len(stops) - 1):
        from_stop, to_stop = stops[i], stops[i + 1]
        from_id, to_id = from_stop.get("id", ""), to_stop.get("id", "")
        from_name, to_name = from_stop.get("name", ""), to_stop.get("name", "")

        if from_id not in target_naptans and to_id not in target_naptans:
          continue

        from_node_id = node_id_map.get(from_id, from_id)
        to_node_id = node_id_map.get(to_id, to_id)

        duration = None
        status_reason = "Success"

        # Step 3: Query TfL Journey Planner API
        jp_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{urllib.parse.quote(from_id)}/to/{urllib.parse.quote(to_id)}"

        try:
          jp_res = requests.get(jp_url, params=params, timeout=5)
          if jp_res.status_code == 200:
            journeys = jp_res.json().get("journeys", [])
            if journeys:
              legs = journeys[0].get("legs", [])
              if legs:
                raw_dur = legs[0].get("duration", journeys[0].get("duration"))
                if raw_dur is not None:
                  duration = float(raw_dur)
            else:
              status_reason = "No journeys returned in response"
          else:
            status_reason = f"HTTP Error {jp_res.status_code}"
        except Exception as err:
          status_reason = f"Request Exception: {err}"

        # Step 4: Fallback to GJT Matrix if API lookup fails or returns 0
        if duration is None or duration <= 0:
          # Try matching via station NLCs if available in all_stations dataframe
          from_row = all_stations[all_stations["NaPTAN"] == from_id]
          to_row = all_stations[all_stations["NaPTAN"] == to_id]
          
          if not from_row.empty and not to_row.empty:
            f_nlc = int(from_row.iloc[0]["NLC"]) if pd.notna(from_row.iloc[0]["NLC"]) else None
            t_nlc = int(to_row.iloc[0]["NLC"]) if pd.notna(to_row.iloc[0]["NLC"]) else None
            if f_nlc and t_nlc and (f_nlc, t_nlc) in gjt_lookup:
              duration = gjt_lookup[(f_nlc, t_nlc)]
              status_reason = "Resolved via GJT Matrix Fallback"

        # Final absolute fallback if GJT also misses
        if duration is None or duration <= 0:
          duration = FALLBACK_TRAVEL_TIME
          status_reason = f"Default Fallback Used ({status_reason})"

        records.append({
            "Line": line_name,
            "From_Node_ID": from_node_id,
            "From_NaPTAN": from_id,
            "From_Station": from_name,
            "To_Node_ID": to_node_id,
            "To_NaPTAN": to_id,
            "To_Station": to_name,
            "Travel_Time_Mins": duration,
            "Status": status_reason
        })

        time.sleep(0.02)  # Rate limiting safeguard

  # Step 5: Format and export final DataFrame
  df = pd.DataFrame(records).drop_duplicates(subset=["Line", "From_Node_ID", "To_Node_ID"])
  
  cols = [
      "Line",
      "From_Node_ID",
      "From_NaPTAN",
      "From_Station",
      "To_Node_ID",
      "To_NaPTAN",
      "To_Station",
      "Travel_Time_Mins",
      "Status"
  ]
  travel_times = df[[c for c in cols if c in df.columns]]

  print(f"\nSuccessfully generated adjacent travel times with {len(travel_times):,} station pairs.")
  travel_times.to_csv("adjacent_station_travel_times.csv", index=False)
  return travel_times


travel_times = get_line_travel_times()
travel_times.head()

Loaded 99,227 pairs into GJT fallback lookup matrix.
Loaded 1,034 unique NaPTAN lookup mappings from all_stations table.
Found 44 rail lines. Processing sequence pairs...
  -> Processing line: Avanti West Coast
  -> Processing line: Bakerloo
  -> Processing line: c2c
  -> Processing line: Central
  -> Processing line: Chiltern Railways
  -> Processing line: Circle
  -> Processing line: CrossCountry
  -> Processing line: District
  -> Processing line: DLR
  -> Processing line: East Midlands Railway
  -> Processing line: Elizabeth line
  -> Processing line: Gatwick Express
  -> Processing line: Grand Central
  -> Processing line: Greater Anglia
  -> Processing line: Great Northern
  -> Processing line: Great Western Railway
  -> Processing line: Hammersmith & City
  -> Processing line: Heathrow Express
  -> Processing line: Hull Trains
  -> Processing line: Island Line
  -> Processing line: Jubilee
  -> Processing line: Liberty
  -> Processing line: Lioness
  -> Processing line: London N

,Line,From_Node_ID,From_NaPTAN,From_Station,To_Node_ID,To_NaPTAN,To_Station,Travel_Time_Mins,Status
0,Avanti West Coast,910GMKNSCEN,910GMKNSCEN,Milton Keynes Central Rail Station,watford junction_west-midlands-trains_1402,910GWATFDJ,Watford Junction Rail Station,0.5,Default Fallback Used (Request Exception: HTTP...
1,Avanti West Coast,watford junction_west-midlands-trains_1402,910GWATFDJ,Watford Junction Rail Station,euston_west-midlands-trains_1444,910GEUSTON,London Euston Rail Station,28.0,Success
2,Avanti West Coast,euston_west-midlands-trains_1444,910GEUSTON,London Euston Rail Station,watford junction_west-midlands-trains_1402,910GWATFDJ,Watford Junction Rail Station,22.0,Success
3,Avanti West Coast,watford junction_west-midlands-trains_1402,910GWATFDJ,Watford Junction Rail Station,910GMKNSCEN,910GMKNSCEN,Milton Keynes Central Rail Station,20.0,Success
4,Bakerloo,harrow & wealdstone_bakerloo_597,940GZZLUHAW,Harrow & Wealdstone Underground Station,kenton_bakerloo_620,940GZZLUKEN,Kenton Underground Station,2.0,Success


## Network Edges: Expected Line Wait Times

### Data Source & Methodology:
* **Source:** Extracted dynamically using the TfL Timetable API (`/Line/{id}/Timetable/{stop}`) at designated benchmark hub stations (`LINE_BENCHMARK_STATIONS`).

* **Assumptions & Fallbacks:** 
  * Passenger arrivals are assumed to be uniform and random.
  * Expected passenger wait time is computed using the half-headway rule: $\text{Wait} = \frac{60}{2 \times \text{TPH}}$ (Trains Per Hour), measuring peak arrivals at hour 08:00–09:00 and off-peak at 11:00–12:00.
  * Where timetable feeds are unavailable or fail, rule-based mode-level headways are applied as explicit fallbacks (e.g., Tube/DLR: 2.5m, Elizabeth Line: 2.0m, Overground: 5.0m, National Rail: 7.5m).

In [ ]:
# ==============================================================================
# LINE WAIT TIMES BUILDER & HALF-HEADWAY LOGIC
# ==============================================================================
# Queries TfL Timetable API for benchmark hub stations to calculate scheduled 
# Trains Per Hour (TPH) during peak and off-peak windows, falling back to 
# mode-level frequency rules where necessary.
# ==============================================================================

# Benchmark hub stations used to calculate line frequencies (TPH)
LINE_BENCHMARK_STATIONS = {
    "victoria": "940GZZLUOXC",  # Oxford Circus
    "jubilee": "940GZZLUBST",  # Baker Street
    "northern": "940GZZLUBNK",  # Bank
    "central": "940GZZLUOXC",  # Oxford Circus
    "bakerloo": "940GZZLUBST",  # Baker Street
    "piccadilly": "940GZZLUGPK",  # Green Park
    "district": "940GZZLUWMS",  # Westminster
    "circle": "940GZZLUBST",  # Baker Street
    "hammersmith-city": "940GZZLUBST", # Baker Street
    "metropolitan": "940GZZLUBST", # Baker Street
    "waterloo-city": "940GZZLUWLO",  # Waterloo
    "dlr": "940GZZDLBNK",  # Bank DLR
    "elizabeth": "940GZZLUTCR",  # Tottenham Court Road
}


def calculate_wait_from_tph(trains_per_hour: float) -> float:
  """Computes expected passenger wait time using the half-headway rule: Wait = 60 / (2 * TPH)."""
  if trains_per_hour <= 0:
    return 10.0
  return round(60.0 / (2.0 * trains_per_hour), 1)


def fetch_scheduled_tph_from_api(line_id: str, stop_naptan: str) -> dict:
  """Queries TfL Timetable API to calculate Peak and Off-Peak Trains Per Hour (TPH)."""
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
  url = f"https://api.tfl.gov.uk/Line/{line_id}/Timetable/{stop_naptan}"

  try:
    res = requests.get(url, params=params, timeout=5)
    if res.status_code == 200:
      data = res.json()
      # Parse scheduled departures in peak (08:00-09:00) and off-peak (11:00-12:00)
      routes = data.get("timetable", {}).get("routes", [])
      peak_count = 0
      off_peak_count = 0

      for route in routes:
        for schedule in route.get("schedules", []):
          for entry in schedule.get("knownJourneys", []):
            hour = int(entry.get("hour", 0))
            if hour == 8:
              peak_count += 1
            elif hour == 11:
              off_peak_count += 1

      if peak_count > 0:
        tph_peak = max(2.0, float(peak_count))
        tph_offpeak = max(1.0, float(off_peak_count))
        return {
            "peak_wait": calculate_wait_from_tph(tph_peak),
            "offpeak_wait": calculate_wait_from_tph(tph_offpeak),
            "default_wait": calculate_wait_from_tph(
                (tph_peak + tph_offpeak) / 2.0
            ),
            "source": "TfL Timetable API (Scheduled TPH)",
        }
  except Exception:
    pass

  return None


def refresh_wait_times():
  """Generates a wait times table using dynamic API frequency extraction with published fallback rules."""
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
  modes = ["tube", "dlr", "overground", "elizabeth-line", "national-rail"]
  url = f"https://api.tfl.gov.uk/Line/Mode/{','.join(modes)}"

  rows = []
  res = requests.get(url, params=params)

  if res.status_code == 200:
    lines_data = res.json()

    for item in lines_data:
      line_name = item.get("name", "").strip()
      mode = item.get("modeName", "").strip()
      line_id = item.get("id", "").strip().lower()

      # 1. Try dynamic API Timetable extraction
      naptan = LINE_BENCHMARK_STATIONS.get(line_id)
      api_metrics = (
          fetch_scheduled_tph_from_api(line_id, naptan) if naptan else None
      )

      if api_metrics:
        peak_w = api_metrics["peak_wait"]
        offpeak_w = api_metrics["offpeak_wait"]
        default_w = api_metrics["default_wait"]
        src = api_metrics["source"]
      else:
        # 2. Rule-based fallbacks (Mode-level average headways)
        if mode == "tube":
          peak_w, offpeak_w, default_w = 2.0, 3.5, 2.5
        elif mode == "dlr":
          peak_w, offpeak_w, default_w = 2.0, 3.5, 2.5
        elif mode == "elizabeth-line":
          peak_w, offpeak_w, default_w = 1.5, 2.5, 2.0
        elif mode == "overground":
          peak_w, offpeak_w, default_w = 4.0, 6.0, 5.0
        else:  # National Rail
          peak_w, offpeak_w, default_w = 5.0, 10.0, 7.5

        src = f"TfL Mode Frequency Rule ({mode.title()})"

      rows.append({
          "Line": line_name,
          "Mode": mode,
          "Peak Avg Wait (mins)": peak_w,
          "Off-Peak Avg Wait (mins)": offpeak_w,
          "Default Avg Wait (mins)": default_w,
          "Data_Source": src,
      })

  wait_times = (
      pd.DataFrame(rows).sort_values(by=["Mode", "Line"]).reset_index(drop=True)
  )

  print(f"Successfully generated {len(wait_times):,} line wait time profiles.")
  return wait_times


wait_times = refresh_wait_times()
wait_times.head()

## Network Edges: Interchange Walking Penalties

### Data Source & Methodology:
* **Source:** Combines published TfL Pay As You Go Out-of-Station Interchange (OSI) timeout matrix (`osi_matrix.csv`) with synthetic API journey triads for in-station platform transfers.

* **Assumptions & Fallbacks:** 
  * OSI fare timeout limits are mapped to physical walking durations ($\le 10\text{m} \to 4.0\text{m}$, $\le 15\text{m} \to 5.0\text{m}$, etc.).

  * **In-Station Interchanges:** Evaluated using a 4-tier hierarchy:
    1. *Empirical Layout Overrides* (e.g., Bank, King's Cross, Waterloo complex hubs).
    2. *Cross-Platform Benchmarks* (2.0m flat penalty for same-platform transfers).
    3. *Synthetic API Triads* (${\text{Walk} = \text{Journey}_{A \to C} - (\text{Leg}_A + \text{Leg}_B + \text{Wait}_B)}$).
    4. *Depth-Tier Heuristics* (Fallback rule based on surface, subsurface, and deep tube layering).

  * **OSI Exclusion:** Out-of-station interchanges will be skipped when building the strict in-station routing graph.

In [ ]:
# ==============================================================================
# SYSTEMATIC INTERCHANGE WALTING PENALTIES & CLASSIFICATION BUILDER
# ==============================================================================
# Categorizes station transfers into Behind-the-Gate In-Station Interchanges, 
# Gated OSIs, and Ungated OSIs using NLC counts and GATED status from all_stations.csv.
# ==============================================================================


def convert_osi_timeout_to_walk_mins(timeout_mins: float) -> float:
  """Converts TfL PAYG revenue timeout limits into physical walking durations."""
  timeout_mins = float(timeout_mins)
  if timeout_mins <= 10:
    return 4.0
  elif timeout_mins <= 15:
    return 5.0
  elif timeout_mins <= 20:
    return 7.0
  elif timeout_mins <= 30:
    return 8.5
  else:
    return 10.0


# Depth tier fallbacks for physical platform walking durations
LINE_DEPTH_TIERS = {
    "Surface/Rail": {
        "National Rail",
        "London Overground",
        "c2c",
        "Chiltern Railways",
        "Southeastern",
        "Thameslink",
        "DLR",
        "Great Northern",
        "Great Western Railway",
        "South Western Railway",
        "Lioness",
        "Mildmay",
        "Windrush",
        "Weaver",
        "Suffragette",
        "Liberty",
    },
    "Subsurface": {"Circle", "District", "Hammersmith & City", "Metropolitan"},
    "Deep Tube": {
        "Bakerloo",
        "Central",
        "Jubilee",
        "Northern",
        "Piccadilly",
        "Victoria",
        "Waterloo & City",
        "Elizabeth line",
    },
}


def get_line_tier(line_name: str) -> str:
  for tier, lines in LINE_DEPTH_TIERS.items():
    if line_name in lines:
      return tier
  return "Deep Tube"


def get_depth_tier_walk_mins(line_a: str, line_b: str) -> float:
  t_a, t_b = get_line_tier(line_a), get_line_tier(line_b)
  if t_a == t_b:
    return 2.0
  elif {t_a, t_b} == {"Surface/Rail", "Deep Tube"}:
    return 7.0
  elif {t_a, t_b} == {"Subsurface", "Deep Tube"}:
    return 5.0
  elif {t_a, t_b} == {"Surface/Rail", "Subsurface"}:
    return 4.0
  return 3.0


# Strict behind-the-gate DLR multi-line hub overrides (where distinct lines share NLCs)
STRICT_DLR_IN_STATION_HUBS = {
    513,  # Bank / Monument Unified
    719,  # Stratford
    884,  # Canning Town
    5145,  # Greenwich
}


def generate_systematic_interchanges():
  df_stations = all_stations.copy()
  df_stations["NLC_int"] = (
      pd.to_numeric(df_stations["NLC"], errors="coerce")
      .dropna()
      .astype(int)
  )

  # Station name NLC multiplicity map
  station_nlc_counts = df_stations.groupby("STATION")["NLC_int"].nunique().to_dict()

  # NLC Gated status map ("gated" vs "ungated")
  nlc_gated_map = dict(
      zip(
          df_stations["NLC_int"],
          df_stations["GATED"].str.strip().str.lower().fillna("gated"),
      )
  )

  nlc_name_map = dict(
      zip(df_stations["NLC_int"], df_stations["STATION"].str.strip())
  )

  df_travel = travel_times
  df_wait = wait_times
  df_osi = pd.read_csv("osi_matrix.csv")

  wait_dict = dict(
      zip(
          df_wait["Line"].str.strip(),
          df_wait["Default Avg Wait (mins)"].astype(float),
      )
  )

  records = []

  # STEP 1: Process Published Out-of-Station Interchanges (OSIs) from Report
  for _, row in df_osi.iterrows():
    try:
      nlc_a = int(row["Station_A"])
      nlc_b = int(row["Station_B"])

      if nlc_b == 0 or nlc_a == nlc_b:
        continue

      raw_timeout = float(row["InterchangeTimeAB"])
      walk_mins = convert_osi_timeout_to_walk_mins(raw_timeout)

      st_a = nlc_name_map.get(nlc_a, str(row["Station_A_Name"]).strip())
      st_b = nlc_name_map.get(nlc_b, str(row["Station_B_Name"]).strip())
      combined_name = f"{st_a} / {st_b}"

      lines_a = df_travel[
          (df_travel["From_NLC"] == nlc_a) | (df_travel["To_NLC"] == nlc_a)
      ]["Line"].unique()
      lines_b = df_travel[
          (df_travel["From_NLC"] == nlc_b) | (df_travel["To_NLC"] == nlc_b)
      ]["Line"].unique()

      gated_a = nlc_gated_map.get(nlc_a, "gated")
      gated_b = nlc_gated_map.get(nlc_b, "gated")
      ic_type = (
          "Out-of-Station Interchange (OSI)"
          if (gated_a == "gated" and gated_b == "gated")
          else "Ungated Out-of-Station Interchange"
      )

      for l_a in lines_a:
        for l_b in lines_b:
          records.append({
              "station": combined_name,
              "station_NLC": nlc_a,
              "From_Line": l_a.strip(),
              "To_Line": l_b.strip(),
              "walk_interchange_times": walk_mins,
              "interchange_type": ic_type,
              "source": "Published TfL OSI Matrix",
          })
    except Exception:
      continue

  # STEP 2: Systematic In-Station vs OSI Classification for Multi-Line Hubs
  from_df = df_travel[["From_NLC", "From_Station", "Line"]].rename(
      columns={"From_NLC": "NLC", "From_Station": "Station"}
  )
  to_df = df_travel[["To_NLC", "To_Station", "Line"]].rename(
      columns={"To_NLC": "NLC", "To_Station": "Station"}
  )
  station_lines = pd.concat([from_df, to_df]).drop_duplicates()

  multi_line_hubs = (
      station_lines.groupby("NLC")["Line"].nunique().loc[lambda x: x > 1].index
  )

  for nlc in multi_line_hubs:
    sub = station_lines[station_lines["NLC"] == nlc]
    st_name = sub["Station"].iloc[0].strip()
    lines = sub["Line"].tolist()
    unique_lines = list(set([l.strip() for l in lines]))

    mapped_nlc = NLC_ALIASES.get(int(nlc), int(nlc))
    num_nlcs_at_station = station_nlc_counts.get(st_name, 1)
    gated_status = nlc_gated_map.get(int(nlc), "gated")

    for i in range(len(unique_lines)):
      for j in range(i + 1, len(unique_lines)):
        l_a, l_b = unique_lines[i], unique_lines[j]

        # Systematic Classification Rule:
        # If single NLC station OR whitelisted behind-the-gate hub -> In-Station Interchange
        # Otherwise -> Out-of-Station Interchange (OSI) or Concourse Transfer
        is_dlr_involved = (
            "dlr" in l_a.lower()
            or "dlr" in l_b.lower()
            or "dlr" in st_name.lower()
        )

        if num_nlcs_at_station == 1 or mapped_nlc in STRICT_DLR_IN_STATION_HUBS:
          ic_type = "In-Station Interchange"
          source = "Systematic Rule (Single NLC / Gated Hub)"
          walk_mins = 2.0  # Default flat cross-platform walk
        elif gated_status == "ungated":
          ic_type = "Ungated Out-of-Station Interchange"
          source = "Systematic Rule (Ungated Node)"
          walk_mins = 4.0
        else:
          ic_type = "Out-of-Station Interchange (OSI)"
          source = "Systematic Rule (Multi-NLC Barrier Separation)"
          walk_mins = get_depth_tier_walk_mins(l_a, l_b)

        records.append({
            "station": st_name,
            "station_NLC": int(nlc),
            "From_Line": l_a,
            "To_Line": l_b,
            "walk_interchange_times": walk_mins,
            "interchange_type": ic_type,
            "source": source,
        })

  # STEP 3: Deduplicate & Enforce Bidirectional Symmetry
  df_out = pd.DataFrame(records).drop_duplicates(
      subset=["station_NLC", "From_Line", "To_Line"]
  )

  reverse_records = []
  for _, row in df_out.iterrows():
    reverse_records.append({
        "station": row["station"],
        "station_NLC": row["station_NLC"],
        "From_Line": row["To_Line"],
        "To_Line": row["From_Line"],
        "walk_interchange_times": row["walk_interchange_times"],
        "interchange_type": row["interchange_type"],
        "source": row["source"],
    })

  df_final = pd.concat([df_out, pd.DataFrame(reverse_records)]).drop_duplicates(
      subset=["station_NLC", "From_Line", "To_Line"]
  )

  cols = [
      "station",
      "station_NLC",
      "From_Line",
      "To_Line",
      "walk_interchange_times",
      "interchange_type",
      "source",
  ]
  interchange_times = df_final[cols].sort_values(
      by=["station", "From_Line", "To_Line"]
  )

  print(
      "Successfully generated"
      f" {len(interchange_times):,} systematically classified interchange"
      " edges."
  )
  print("\nBreakdown by Interchange Type:")
  print(interchange_times["interchange_type"].value_counts())

  return interchange_times

interchange_times = generate_systematic_interchanges()
interchange_times.head()

## Network Graph Construction (`NetworkX`) with NLC Aliasing

Constructs a directed graph representing the London contactless transit network:

* **NLC Aliasing:** Harmonizes legacy, variant, and adjacent station codes (e.g., Monument $\to$ Bank, Bow Road $\to$ Bow Church) using `NLC_ALIASES` to ensure methodological consistency.

* **Nodes:** Represent unique physical gate lines (`NLC_Line`), keeping multi-modal platforms (London Bridge NR vs. Underground) separate.

* **Edges (OSIs Excluded):** Journey travel times, boarding wait times, and in-station interchange walking penalties.

In [ ]:
# ==============================================================================
# NETWORK GRAPH CONSTRUCTION & DIJKSTRA TOPOLOGY BUILDER (WITH NLC ALIASES)
# ==============================================================================
# Constructs a directed NetworkX graph with NLC aliasing applied to nodes and edges.
# Out-of-Station Interchanges (OSIs) are strictly excluded.
# ==============================================================================

print("Initializing NetworkX Directed Graph...")
G = nx.DiGraph()

# Mapping of legacy/variant NLCs and adjacent station stops to primary DLR/hub NLCs
NLC_ALIASES = {
    # Legacy / Variant DLC & Rail NLC Unification
    843: 884,  # Canning Town DLR -> Canning Town Primary NLC
    573: 5152,  # Woolwich Arsenal DLR -> Woolwich Arsenal Primary NLC
    844: 513,  # Bank DLR -> Bank Primary NLC
    863: 719,  # Stratford DLR -> Stratford Primary NLC
    6960: 719,  # Stratford NR -> Stratford Primary NLC
    1082: 860,  # Shadwell LO -> Shadwell Primary NLC
    842: 852,  # Canary Wharf DLR -> Canary Wharf Primary NLC
    887: 6561,  # Custom House DLR -> Custom House Primary NLC
    928: 5145,  # Greenwich DLR -> Greenwich Primary NLC
    862: 7491,  # Limehouse DLR -> Limehouse Primary NLC
    940: 5051,  # Lewisham DLR -> Lewisham Primary NLC
    559: 889,  # West Silvertown DLR -> West Silvertown Primary NLC

    # Adjacent Station / Cross-Platform Mappings (Consistency with Manual Review)
    528: 851,  # Bow Road LU -> Bow Church DLR
    644: 513,  # Monument LU -> Bank Unified Hub
    731: 888,  # Tower Hill LU -> Tower Gateway DLR
}

# 1. Build lookup maps and apply NLC_ALIASES to station NLCs
df_stations = all_stations.copy()
df_stations["NLC_int"] = pd.to_numeric(
    df_stations["NLC"], errors="coerce"
).astype("Int64")

# Map original NLC to aliased primary NLC
nlc_to_name = {}
for _, row in df_stations.dropna(subset=["NLC_int"]).iterrows():
  orig_nlc = int(row["NLC_int"])
  mapped_nlc = NLC_ALIASES.get(orig_nlc, orig_nlc)
  station_name = str(row["STATION"]).strip()
  nlc_to_name[mapped_nlc] = station_name

# Identify DLR stations using mapped NLCs
dlr_stations_df = df_stations[
    df_stations["STATION"].str.contains("DLR", case=False, na=False)
]
dlr_nlcs = set(
    dlr_stations_df["NLC_int"]
    .dropna()
    .astype(int)
    .map(lambda x: NLC_ALIASES.get(x, x))
)

print(
    f"Loaded {len(nlc_to_name):,} unique station nodes ({len(dlr_nlcs)} DLR"
    " endpoint groups)."
)

# 2. Add Journey Edges (with NLC Aliasing applied)
travel_added = 0
for _, row in travel_times.iterrows():
  try:
    orig_from_nlc = int(row["From_NLC"])
    orig_to_nlc = int(row["To_NLC"])

    # Apply NLC Aliasing
    from_nlc = NLC_ALIASES.get(orig_from_nlc, orig_from_nlc)
    to_nlc = NLC_ALIASES.get(orig_to_nlc, orig_to_nlc)

    line = str(row["Line"]).strip()
    duration = float(row["Travel_Time_Mins"])

    suffix = "DLR" if line.lower() == "dlr" else line
    u_node = f"{from_nlc}_{suffix}"
    v_node = f"{to_nlc}_{suffix}"

    G.add_edge(u_node, v_node, weight=duration, edge_type="journey", line=line)
    G.add_edge(v_node, u_node, weight=duration, edge_type="journey", line=line)
    travel_added += 2
  except Exception:
    continue

print(f"Added {travel_added:,} directional journey edges.")

# 3. Add Wait Edges (Initial boarding wait times)
wait_dict = dict(
    zip(
        wait_times["Line"].str.strip(),
        wait_times["Default Avg Wait (mins)"].astype(float),
    )
)
wait_added = 0
for nlc in nlc_to_name.keys():
  station_lines = travel_times[
      (travel_times["From_NLC"].map(lambda x: NLC_ALIASES.get(int(x), int(x)))
       == nlc)
      | (
          travel_times["To_NLC"].map(
              lambda x: NLC_ALIASES.get(int(x), int(x))
          )
          == nlc
      )
  ]["Line"].unique()

  for line in station_lines:
    suffix = "DLR" if str(line).lower() == "dlr" else str(line)
    node_id = f"{nlc}_{suffix}"
    if G.has_node(node_id):
      wait_time = wait_dict.get(str(line).strip(), 2.5)
      G.add_edge(
          node_id, node_id, weight=wait_time, edge_type="wait", line=line
      )
      wait_added += 1

print(f"Added {wait_added:,} station boarding wait edges.")

# 4. Add Interchange Edges (In-station platform transfers, OSIs strictly excluded)
interchange_added = 0
for _, row in interchange_times.iterrows():
  try:
    if "OSI" in str(row["interchange_type"]):
      continue  # Skip OSIs

    orig_nlc = int(row["station_NLC"])
    nlc = NLC_ALIASES.get(orig_nlc, orig_nlc)

    l_from = str(row["From_Line"]).strip()
    l_to = str(row["To_Line"]).strip()
    walk_time = float(row["walk_interchange_times"])

    s_from = "DLR" if l_from.lower() == "dlr" else l_from
    s_to = "DLR" if l_to.lower() == "dlr" else l_to

    u_node = f"{nlc}_{s_from}"
    v_node = f"{nlc}_{s_to}"

    if G.has_node(u_node) and G.has_node(v_node):
      G.add_edge(
          u_node,
          v_node,
          weight=walk_time,
          edge_type="interchange",
          line=f"{l_from}->{l_to}",
      )
      interchange_added += 1
  except Exception:
    continue

print(
    "Added"
    f" {interchange_added:,} in-station interchange edges (OSIs excluded)."
)
print(
    "\nGraph Construction Complete: Total Nodes ="
    f" {len(G.nodes):,} | Total Edges = {len(G.edges):,}"
)

## Serialize & Save Network Graph

Saves the in-memory NetworkX directed graph and associated lookup dictionaries (`nlc_to_name`, `df_stations`) to disk (`network_graph_no_osi.pkl`). This allows subsequent routing and auditing cells to load the network topology instantly without rebuilding it.

In [ ]:
# ==============================================================================
# SERIALIZE & SAVE NETWORK GRAPH
# ==============================================================================
# Packages the NetworkX graph and lookup metadata into a dictionary and 
# serializes it using pickle for downstream routing scripts and notebooks.
# ==============================================================================

# Bundle graph and metadata structures
graph_data = {
    "graph": G,
    "nlc_to_name": nlc_to_name,
    "stations": df_stations,
}

output_filename = "network_graph.pkl"

with open(output_filename, "wb") as f:
  pickle.dump(graph_data, f)

print(
    f"Successfully serialized and saved network graph to '{output_filename}'"
    f" ({os.path.getsize(output_filename) / (1024 * 1024):.2f} MB)."
)

## DLR Routing Engine

Executes Dijkstra shortest-path routing across the network graph from all non-DLR station gate lines to all DLR station endpoints (and vice versa):

* **In-Gate Routing Only:** OSIs are excluded, ensuring only valid platform-to-platform transfers are used.

* **DLR Interchange Identification:** Traces each shortest path to identify the exact node where the journey transitions onto or off the DLR network.

* **Output Schema:** Produces a comprehensive CSV table (`dlr_journey_interchanges.csv`) containing origin/destination names, NLCs, designated DLR interchanges, full interchange station lists, and formatted journey times.

In [ ]:
# ==============================================================================
# BATCH DLR ROUTING ENGINE & INTERCHANGE AUDIT BUILDER
# ==============================================================================
# Computes shortest paths between all non-DLR and DLR NLC gate lines,
# extracts transfer transition points, and formats the output into the 
# required 8-column schema.
# ==============================================================================


def get_route_interchanges_and_dlr_point(path, nlc_to_name):
  """Traces a NetworkX path to identify all interchange stations

  and the specific DLR interchange transition point.
  """
  dlr_interchange_nlc = None
  dlr_interchange_name = None
  all_interchanges = []

  for i in range(len(path) - 1):
    u_node, v_node = path[i], path[i + 1]
    u_parts = u_node.split("_")
    v_parts = v_node.split("_")

    u_nlc, u_line = int(u_parts[0]), u_parts[1]
    v_nlc, v_line = int(v_parts[0]), v_parts[1]

    # Detect line transition (Interchange)
    if u_line != v_line and u_nlc == v_nlc:
      mapped_nlc = NLC_ALIASES.get(u_nlc, u_nlc)
      st_name = nlc_to_name.get(mapped_nlc, f"NLC_{mapped_nlc}")
      if st_name not in all_interchanges:
        all_interchanges.append(st_name)

    # Detect DLR transition (Onto or off DLR)
    if (u_line == "DLR" and v_line != "DLR") or (
        u_line != "DLR" and v_line == "DLR"
    ):
      transition_nlc = u_nlc if u_line == "DLR" else v_nlc
      mapped_trans_nlc = NLC_ALIASES.get(transition_nlc, transition_nlc)
      dlr_interchange_nlc = mapped_trans_nlc
      dlr_interchange_name = nlc_to_name.get(
          mapped_trans_nlc, f"NLC_{mapped_trans_nlc}"
      )

  # If route is entirely DLR or direct without explicit boundary node capture
  if not dlr_interchange_name and path:
    for node in path:
      parts = node.split("_")
      if parts[1] == "DLR":
        raw_nlc = int(parts[0])
        mapped_nlc = NLC_ALIASES.get(raw_nlc, raw_nlc)
        dlr_interchange_nlc = mapped_nlc
        dlr_interchange_name = nlc_to_name.get(
            mapped_nlc, f"NLC_{mapped_nlc}"
        )
        break

  all_interchanges_str = (
      ", ".join(all_interchanges) if all_interchanges else "Direct / None"
  )
  return dlr_interchange_nlc, dlr_interchange_name, all_interchanges_str


def run_batch_dlr_routing():
  # Identify DLR nodes and NLCs
  dlr_nodes = [n for n in G.nodes if n.endswith("_DLR")]
  dlr_nlcs = set(int(n.split("_")[0]) for n in dlr_nodes)
  all_nlcs = set(df_stations["NLC"].dropna().astype(int))
  non_dlr_nlcs = all_nlcs - dlr_nlcs

  print(
      f"Running batch routing across {len(dlr_nlcs)} DLR endpoint NLCs and"
      f" {len(non_dlr_nlcs)} non-DLR NLC gate lines..."
  )
  results = []

  # Helper to process and append route results
  def add_route_record(direction, u_nlc, v_nlc, cost, path):
    total_mins = int(round(cost))
    hours, mins = divmod(total_mins, 60)
    journey_time_str = f"{hours}:{mins:02d}"

    u_mapped = NLC_ALIASES.get(u_nlc, u_nlc)
    v_mapped = NLC_ALIASES.get(v_nlc, v_nlc)

    from_station = nlc_to_name.get(u_mapped, f"NLC_{u_mapped}")
    to_station = nlc_to_name.get(v_mapped, f"NLC_{v_mapped}")

    ic_nlc, ic_name, all_ics = get_route_interchanges_and_dlr_point(
        path, nlc_to_name
    )

    results.append({
        "from station": from_station,
        "from station NLC": u_mapped,
        "to station": to_station,
        "to station NLC": v_mapped,
        "dlr interchange": ic_name,
        "dlr interchange NLC": ic_nlc,
        "all interchange stations": all_ics,
        "journey time": journey_time_str,
        "Journey_Time_Mins": total_mins,
        "Direction": direction,
    })

  # 1. Non-DLR -> DLR Journeys
  for u_nlc in non_dlr_nlcs:
    u_nodes = [n for n in G.nodes if n.startswith(f"{u_nlc}_")]
    if not u_nodes:
      continue
    for v_nlc in dlr_nlcs:
      v_nodes = [n for n in G.nodes if n.startswith(f"{v_nlc}_DLR")]
      if not v_nodes:
        continue

      best_cost, best_path = float("inf"), None
      for s in u_nodes:
        for t in v_nodes:
          if nx.has_path(G, s, t):
            c = nx.shortest_path_length(G, s, t, weight="weight")
            if c < best_cost:
              best_cost, best_path = c, nx.shortest_path(
                  G, s, t, weight="weight"
              )

      if best_path:
        add_route_record(
            "Non-DLR to DLR", u_nlc, v_nlc, best_cost, best_path
        )

  # 2. DLR -> Non-DLR Journeys
  for u_nlc in dlr_nlcs:
    u_nodes = [n for n in G.nodes if n.startswith(f"{u_nlc}_DLR")]
    if not u_nodes:
      continue
    for v_nlc in non_dlr_nlcs:
      v_nodes = [n for n in G.nodes if n.startswith(f"{v_nlc}_")]
      if not v_nodes:
        continue

      best_cost, best_path = float("inf"), None
      for s in u_nodes:
        for t in v_nodes:
          if nx.has_path(G, s, t):
            c = nx.shortest_path_length(G, s, t, weight="weight")
            if c < best_cost:
              best_cost, best_path = c, nx.shortest_path(
                  G, s, t, weight="weight"
              )

      if best_path:
        add_route_record(
            "DLR to Non-DLR", u_nlc, v_nlc, best_cost, best_path
        )

  # Format output DataFrame
  df_out = pd.DataFrame(results).drop_duplicates(
      subset=["from station NLC", "to station NLC"]
  )

  cols = [
      "from station",
      "from station NLC",
      "to station",
      "to station NLC",
      "dlr interchange",
      "dlr interchange NLC",
      "all interchange stations",
      "journey time",
  ]
  master_routes_df = df_out[cols]

  master_routes_df.to_csv("dlr_journey_interchanges.csv", index=False)
  print(
      "\nRouting Complete: Successfully generated"
      f" {len(master_routes_df):,} total route pairs in"
      " 'dlr_journey_interchanges.csv'."
  )
  return master_routes_df

master_routes_df = run_batch_dlr_routing()
master_routes_df.head()

In [ ]:
# ==============================================================================
# DLR ROUTING AUDIT & COMPARISON SCRIPT
# ==============================================================================
# Compares 'dlr_journey_interchanges.csv' against the manual review baseline.
# ==============================================================================

import os
import pandas as pd

# 1. Load datasets
manual_file = "DLR interchange manual review.csv"
auto_file = "dlr_journey_interchanges.csv"

if not os.path.exists(manual_file) or not os.path.exists(auto_file):
  print(
      "Error: Required CSV files not found. Ensure both files are in the"
      " directory."
  )
else:
  df_manual = pd.read_csv(manual_file)
  df_auto = pd.read_csv(auto_file)

  # Standardize column names
  df_manual.columns = df_manual.columns.str.strip()
  df_auto.columns = df_auto.columns.str.strip()

  # Master NLC Alias Unification Map (to bridge old/new and adjacent stations)
  NLC_ALIASES = {
      843: 884,  # Canning Town
      573: 5152,  # Woolwich Arsenal
      844: 513,  # Bank
      863: 719,  # Stratford
      6960: 719,  # Stratford NR
      1082: 860,  # Shadwell LO
      842: 852,  # Canary Wharf
      887: 6561,  # Custom House
      928: 5145,  # Greenwich
      862: 7491,  # Limehouse
      940: 5051,  # Lewisham
      559: 889,  # West Silvertown
      528: 851,  # Bow Road LU -> Bow Church DLR
      644: 513,  # Monument LU -> Bank Unified Hub
      731: 888,  # Tower Hill LU -> Tower Gateway DLR
  }

  # Apply NLC mapping
  df_manual["NLC1_mapped"] = (
      pd.to_numeric(df_manual["NLC1"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )
  df_manual["NLC2_mapped"] = (
      pd.to_numeric(df_manual["NLC2"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )

  df_auto["from_NLC_mapped"] = (
      pd.to_numeric(df_auto["from station NLC"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )
  df_auto["to_NLC_mapped"] = (
      pd.to_numeric(df_auto["to station NLC"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )

  # Deduplicate
  m_uniq = df_manual.drop_duplicates(subset=["NLC1_mapped", "NLC2_mapped"]).copy()
  a_uniq = df_auto.drop_duplicates(
      subset=["from_NLC_mapped", "to_NLC_mapped"]
  ).copy()

  # Metric 1: Total number of routes and unique NLC stations
  print("=== 1. Total Routes & Unique NLC Stations ===")
  print(f"Total Unique Routes in Manual Review: {len(m_uniq):,}")
  print(f"Total Unique Routes in Automated Graph: {len(a_uniq):,}")

  manual_nlcs = set(m_uniq["NLC1_mapped"]).union(set(m_uniq["NLC2_mapped"]))
  auto_nlcs = set(a_uniq["from_NLC_mapped"]).union(
      set(a_uniq["to_NLC_mapped"])
  )
  print(f"Unique NLC Gate Lines in Manual Review: {len(manual_nlcs):,}")
  print(f"Unique NLC Gate Lines in Automated Graph: {len(auto_nlcs):,}")

  # Metric 2: Distribution of DLR Interchanges Found
  print("\n=== 2. Distribution of DLR Interchanges Found (Top Hubs) ===")
  print("\nManual Review DLR Interchanges:")
  print(m_uniq["DLR_INT Name"].value_counts().head(10))

  print("\nAutomated Graph DLR Interchanges:")
  print(a_uniq["dlr interchange"].value_counts().head(10))

  # Metric 3: Routes with no interchanges vs interchanges present
  m_no_ic = m_uniq["DLR_INT"].isna() | (m_uniq["DLR_INT"] == 0)
  a_no_ic = (
      a_uniq["dlr interchange"].isna()
      | (a_uniq["dlr interchange"] == "Direct / None")
      | (a_uniq["dlr interchange"] == "")
  )

  print("\n=== 3. Interchange Presence Comparison ===")
  print(f"Manual Routes with No DLR Interchange Logged: {m_no_ic.sum():,}")
  print(f"Automated Routes with No DLR Interchange Found: {a_no_ic.sum():,}")

  # Merge datasets for direct overlap analysis
  merged = pd.merge(
      m_uniq,
      a_uniq,
      left_on=["NLC1_mapped", "NLC2_mapped"],
      right_on=["from_NLC_mapped", "to_NLC_mapped"],
      how="outer",
      indicator=True,
  )

  print("\n=== 4. Dataset Overlap & Agreement ===")
  merge_counts = merged["_merge"].value_counts()
  print(f"Shared O-D Pairs (Matched Both): {merge_counts.get('both', 0):,}")
  print(f"Manual Review Only: {merge_counts.get('left_only', 0):,}")
  print(f"Automated Graph Only: {merge_counts.get('right_only', 0):,}")

In [ ]:
# Diagnostic: Find NLCs present in manual review but missing from automated graph
manual_nlcs = set(df_manual['NLC1']).union(set(df_manual['NLC2']))
auto_nlcs = set(df_auto['from station NLC']).union(
    set(df_auto['to station NLC'])
)

missing_nlcs = manual_nlcs - auto_nlcs
print(f'Number of NLCs in manual review missing from automated graph: {len(missing_nlcs)}')

In [ ]:
# ==============================================================================
# DIAGNOSTIC: FIND STATIONS IN all_stations.csv MISSING FROM line_travel_times.csv
# ==============================================================================

# 1. Load datasets
df_stations = pd.read_csv("all_stations.csv")
df_travel = pd.read_csv("line_travel_times.csv")

# 2. Extract unique NLC sets
all_station_nlcs = set(
    pd.to_numeric(df_stations["NLC"], errors="coerce").dropna().astype(int)
)
travel_nlcs = set(
    pd.to_numeric(df_travel["From_NLC"], errors="coerce").dropna().astype(int)
).union(
    set(pd.to_numeric(df_travel["To_NLC"], errors="coerce").dropna().astype(int))
)

# 3. Identify missing NLCs
missing_nlcs = all_station_nlcs - travel_nlcs
df_missing = df_stations[
    df_stations["NLC"].astype(int).isin(missing_nlcs)
].copy()
df_missing.to_csv("stations_missing_from_travel_times.csv", index=False)

# 4. Print summary metrics
print(f"Total stations in all_stations.csv: {len(df_stations):,}")
print(
    "Stations with valid travel time edges:"
    f" {len(all_station_nlcs) - len(missing_nlcs):,}"
)
print(f"Stations missing from travel times: {len(df_missing):,}\n")

# Display missing stations grouped by transport operator/joint network
if not df_missing.empty:
  print("Breakdown of missing stations by network joint type:")
  print(df_missing["JOINT"].value_counts())
  print("\nFirst 15 missing stations:")
  display(df_missing[["NLC", "STATION", "JOINT", "GATED"]])
else:
  print("All stations in all_stations.csv have travel time edges!")